# AIPix ImageEngine · Colab

این دفترچه یک موتور تصویر مستقل با مدل متن‌باز SDXL Turbo راه‌اندازی می‌کند. مدل به طبیعت محدود نیست و از پرامپت‌های عمومی برای آدم‌ها، حیوانات، ماشین‌ها، اشیا، معماری، فانتزی، انیمه و سبک‌های مختلف استفاده می‌کند.

مدل روی Runtime ابری اجرا می‌شود؛ وزن‌های مدل روی همان Runtime دانلود می‌شوند. دسترسی GPU رایگان و اجرای دائمی تضمین‌شده نیست.

## ۱) بررسی GPU

در Colab از Runtime → Change runtime type گزینه GPU را انتخاب کن، اگر برای حسابت فعال بود.

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('برای اجرای این نسخه به GPU نیاز است. در تنظیمات Runtime گزینه GPU را انتخاب کن.')
print('GPU:', torch.cuda.get_device_name(0))


## ۲) نصب و بارگذاری مدل

SDXL Turbo یک مدل عمومی text-to-image است. برای سرعت، اندازه خروجی ۵۱۲ پیکسل و ۴ مرحله تولید استفاده می‌شود.

In [ ]:
%pip -q install -U diffusers transformers accelerate safetensors fastapi uvicorn


In [ ]:
import torch
from diffusers import AutoPipelineForText2Image

IMAGE_MODEL_ID = 'stabilityai/sdxl-turbo'
pipe = AutoPipelineForText2Image.from_pretrained(
    IMAGE_MODEL_ID,
    torch_dtype=torch.float16,
    variant='fp16',
)
pipe.to('cuda')
pipe.enable_attention_slicing()
print('ImageEngine model ready:', IMAGE_MODEL_ID)


## ۳) ساخت API خصوصی

این سلول یک توکن تصادفی می‌سازد. آن را فقط در تنظیمات برنامه خودت وارد کن و در GitHub عمومی قرار نده.

In [ ]:
import base64, io, secrets, threading, time
from typing import Optional
from fastapi import FastAPI, Header, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field
import uvicorn
from google.colab import output

IMAGE_API_TOKEN = secrets.token_urlsafe(32)
app = FastAPI(title='AIPix ImageEngine')
app.add_middleware(CORSMiddleware, allow_origins=['*'], allow_credentials=False, allow_methods=['GET','POST','OPTIONS'], allow_headers=['Authorization','Content-Type'])

class ImageRequest(BaseModel):
    prompt: str = Field(min_length=2, max_length=2000)
    size: str = '512x512'

@app.get('/health')
def health():
    return {'ok': True, 'model': IMAGE_MODEL_ID, 'status': 'ready'}

@app.post('/v1/images/generations')
def generate(req: ImageRequest, authorization: Optional[str] = Header(default=None)):
    if authorization != f'Bearer {IMAGE_API_TOKEN}':
        raise HTTPException(status_code=401, detail='Invalid API token')
    if req.size != '512x512':
        raise HTTPException(status_code=400, detail='This Colab build currently supports 512x512 only')
    prompt = req.prompt.strip()
    if not prompt:
        raise HTTPException(status_code=400, detail='Prompt cannot be empty')
    try:
        result = pipe(prompt=prompt, num_inference_steps=4, guidance_scale=0.0, height=512, width=512)
        image = result.images[0]
        buffer = io.BytesIO()
        image.save(buffer, format='PNG')
        encoded = base64.b64encode(buffer.getvalue()).decode('ascii')
        return {'created': True, 'data': [{'b64_json': encoded}], 'model': IMAGE_MODEL_ID}
    except Exception as exc:
        raise HTTPException(status_code=500, detail=f'Image generation failed: {exc}')

def run_server():
    uvicorn.run(app, host='0.0.0.0', port=7861, log_level='warning')

threading.Thread(target=run_server, daemon=True).start()
time.sleep(2)
IMAGE_PUBLIC_URL = output.eval_js('google.colab.kernel.proxyPort(7861)')
print('ImageEngine URL:', IMAGE_PUBLIC_URL + '/v1/images/generations')
print('Health:', IMAGE_PUBLIC_URL + '/health')
print('ImageEngine token:', IMAGE_API_TOKEN)


## وضعیت و نکات

- موتور تصویر به یک دسته سوژه خاص محدود نشده است؛ نوع نتیجه به پرامپت و توانایی مدل بستگی دارد.
- این نسخه تصویر ۵۱۲×۵۱۲ می‌سازد و برای سرعت از ۴ مرحله استفاده می‌کند.
- API تا وقتی Runtime روشن باشد فعال است. با قطع Runtime باید دفترچه دوباره اجرا شود و URL و توکن تازه تولید می‌شوند.
- GPU رایگان، سرعت و سهمیه نامحدود تضمین‌شده نیست.
- این Notebook موتور تصویر را جدا از Notebook چت اجرا می‌کند تا حافظه GPU مدل چت و مدل تصویر با هم تداخل نداشته باشند.
